# Chapter 30 — Debugging by Layer

**Companion to *Pi Agents*** · [`Pi Agents` chapter 30](https://programmer.ie/books/pi/30-chapter/)

| | |
|---|---|
| Chapter | `30-chapter.md` · weight 30 · `/books/pi/30-chapter/` |
| Notebook | `notebooks/pi/30-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

For one symptom, which layer does it survive when the layer below is replaced?

The chapter's method: **replace the layer below the one you suspect, and ask
whether the symptom survives.** The layer list and the method are the book's
**proposal**, not a Pi procedure. This notebook demonstrates the method on
machinery that exists: the scripted provider and the session harness.


## A note on what this chapter is

Chapter 30 has **no example directory and no ledger rows**. Its method is the
book's own formulation — a **proposal**, not a Pi contract — and the chapter's
only code fence is marked `documented-not-run`. This notebook therefore
**demonstrates** the method on machinery that does exist, instead of presenting a
scripted result as evidence about Pi.

The chain of thought is the chapter's; the code that runs it is the book's own
harness and the pinned packages.

## What this notebook establishes

- The **replacement test** can be run on real machinery: a scripted provider stands in for the model, and the symptom either survives or disappears.
- A symptom that **survives** a provider replacement is in the code above the model — the tool, the gate, or the loop.
- A symptom that **disappears** when the interface is replaced was presentation.
- The demonstration uses the same `faux` provider and `makeSession` harness as every other chapter; it adds no new mechanism.

## What this notebook does **not** establish

- **The layer list and the method are the book's proposal** (`ch30-lim3`), not documented by Pi.
- **The method is unvalidated against a set of real failures** — it is a heuristic that worked on the failures described, not a procedure.
- **The `onPayload` snippet in the chapter is `documented-not-run`** — it is a real `pi-ai` API, not exercised here.
- **No real model is called** — this demonstrates the method, not a diagnosis of any particular failure.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Demonstrate** + Inspect — the replacement method run on the scripted provider and the session harness. The layer list is the book's proposal.
* **Evidence scope:** `in_process_runtime`. The mechanism (faux provider, sessions) is Pi's; the method is the book's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(30))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/30-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## The suspects

```text
interface              terminal, print, JSON, RPC, SDK: how you watch it
host / application     extensions, settings, trust, context files, sessions
tool / effect          what a tool did, and what a gate let through
agent core             the loop: turns, hooks, events, queues
model access (pi-ai)   one request: auth, payload, stop reason, usage
provider               the service that answered
```

The one question: **replace the layer below the one you suspect. Does the
symptom survive?**

In [3]:
LAYERS = pinb.driver_source("ch30-layers.ts")
layers = pinb.driver(LAYERS, label="ch30-layers", timeout=180)
print(pinb.md_table(
    ["case", "isError?", "text"],
    [["tool throws", layers["symptom"]["throwingTool"]["isError"], layers["symptom"]["throwingTool"]["text"][:40]],
     ["tool returns", layers["symptom"]["returningTool"]["isError"], layers["symptom"]["returningTool"]["text"][:40]],
     ["gate blocks", layers["symptom"]["gate"]["isError"], layers["symptom"]["gate"]["text"][:40]]],
))

| case | isError? | text |
|---|---|---|
| tool throws | True | tool blew up |
| tool returns | False | wrote the file |
| gate blocks | True | gate refused |


In [4]:
pinb.show_checks([
    pinb.check("a throwing tool produces a failed result the model can read",
               layers["symptom"]["throwingTool"]["isError"], "isError: true"),
    pinb.check("a returning tool produces a successful result",
               not layers["symptom"]["returningTool"]["isError"], "isError: false"),
    pinb.check("a gate block produces a failed result",
               layers["symptom"]["gate"]["isError"], "isError: true"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------------  --------------
PASS  a throwing tool produces a failed result the model can read  isError: true
PASS  a returning tool produces a successful result  isError: false
PASS  a gate block produces a failed result  isError: true

3/3 assertions held.


## Step 1: replace the provider

The scripted provider stands in for the model. If the symptom survives with a
scripted provider, the cause is **not the model** — it is the tool, the gate, or
the loop.

In [5]:
r = layers["replaceProvider"]
print("Symptom survives with a scripted provider:", r["symptomSurvivesWithScriptedProvider"])
print("Conclusion:", r["conclusion"])
print()
pinb.show_checks([
    pinb.check("the failure survives the provider replacement",
               r["symptomSurvivesWithScriptedProvider"] is True,
               "symptom is above the model"),
])

Symptom survives with a scripted provider: True
Conclusion: the symptom is in the tool or the gate, not the model

== Assertions ==
  assertion  observed
----  ---------------------------------------------  --------------------------
PASS  the failure survives the provider replacement  symptom is above the model

1/1 assertions held.


## Step 2: replace the interface

The same command, run with a UI and without one. The symptom (a notification)
**disappears** without a UI — it was presentation, not the guard logic.

In [6]:
i = layers["replaceInterface"]
print("With UI: notifications =", i["withUI"])
print("No UI:   notifications =", i["noUI"])
print()
pinb.show_checks([
    pinb.check("with a UI the command notifies",
               len(i["withUI"]) > 0, "notification delivered"),
    pinb.check("without a UI the notification disappears (presentation)",
               len(i["noUI"]) == 0, "no notification"),
    pinb.check("so the symptom was the interface, not the logic",
               i["symptomDisappears"], "interface layer"),
])

With UI: notifications = ['acted (with UI)']
No UI:   notifications = []

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  ----------------------
PASS  with a UI the command notifies  notification delivered
PASS  without a UI the notification disappears (presentation)  no notification
PASS  so the symptom was the interface, not the logic  interface layer

3/3 assertions held.


## The decision diagram

```mermaid
flowchart TD
  S["a symptom"] --> Q1{"Survives with a scripted provider?"}
  Q1 -->|"no"| MODEL["model, credential or request"]
  Q1 -->|"yes"| Q2{"Survives in a bare Agent, no coding agent?"}
  Q2 -->|"no"| HOST["host or application"]
  Q2 -->|"yes"| Q3{"Survives with every hook removed?"}
  Q3 -->|"no"| HOOK["a hook or a gate"]
  Q3 -->|"yes"| CORE["the loop, or your own tools"]
```

*The diagram is the book's proposal. The demonstration above walks two of its
edges with real machinery.*

## Your turn: assign a symptom

**Predict first.** The model "keeps retrying". The tool returns an object saying
`{ failed: true }` rather than setting `isError: true`. Is that a failure the
model sees as failed?

**Then change one input.** In the demonstration, the gate blocks. Remove the
`tool_call` handler and re-run. Which layer's symptom disappears?

**Predict the boundary.** The chapter says the hardest failures are where two
layers each look correct. Give one: an effect with no cause in the transcript.
What else runs with the process's permissions (chapter 42)?

In [7]:
print("Predictions:")
print("  1. { failed: true } without isError: true is NOT a failure to the model.")
print("  2. Remove the gate: the block symptom disappears - it was the gate.")
print("  3. An effect with no transcript cause: an extension, a postinstall, or an MCP server.")
print()
print("Observed above:")
print(f"  throwing tool: isError={layers['symptom']['throwingTool']['isError']}")
print(f"  gate: isError={layers['symptom']['gate']['isError']}")
print(f"  symptom survives provider replacement: {layers['replaceProvider']['symptomSurvivesWithScriptedProvider']}")
print(f"  interface symptom disappears without UI: {layers['replaceInterface']['symptomDisappears']}")
print()
assert layers["symptom"]["throwingTool"]["isError"]
assert not layers["symptom"]["returningTool"]["isError"]
assert layers["replaceProvider"]["symptomSurvivesWithScriptedProvider"]
assert layers["replaceInterface"]["symptomDisappears"]
print("held (demonstration): the replacement test assigns a symptom to a layer on real machinery")

Predictions:
  1. { failed: true } without isError: true is NOT a failure to the model.
  2. Remove the gate: the block symptom disappears - it was the gate.
  3. An effect with no transcript cause: an extension, a postinstall, or an MCP server.

Observed above:
  throwing tool: isError=True
  gate: isError=True
  symptom survives provider replacement: True
  interface symptom disappears without UI: True

held (demonstration): the replacement test assigns a symptom to a layer on real machinery


## Interpretation

The method is one question, asked of each layer in turn:

> **Replace the layer below the one you suspect. Does the symptom survive?**

| To test this layer… | …replace this | And look for |
|---|---|---|
| Your code above the model | the provider, with the faux provider | The symptom survives: it is your code or the loop |
| The agent loop | the coding agent, with a bare `Agent` | The symptom disappears: it is in the host |
| An extension | run with extensions off (`-ne`) | The symptom disappears: it is that extension |
| The interface | `--mode json` or the SDK | The symptom disappears: it was presentation |
| The provider | one request through `pi-ai` directly | The symptom survives with no agent: it is the model |

**Proposed, not documented.** The layer list and the method are this book's
formulation. They follow from the layers being separable (chapters 1 and 37) and
have not been validated against a set of real failures. Use them as a heuristic.

The artefacts cut across layers: `/debug` and `pi-debug.log`, `/bug`, `/session`,
`/hotkeys`, the startup header, and `pi --export`. Review any of them before
sharing — they can contain prompts, responses, tool output and file contents.

## Sources, execution mode and limitations

**Execution mode:** **Demonstrate** + Inspect — the replacement method run on the scripted provider and the session harness. The layer list is the book's proposal.

**Evidence scope:** `in_process_runtime`. The mechanism (faux provider, sessions) is Pi's; the method is the book's.

### What was read or run

- **Ran** `drivers/ch30-layers.ts`: real sessions demonstrating the replacement test (tool throws, tool returns, gate blocks, command with/without UI).
- **Read** `content/books/pi/30-chapter.md` — the layer list, the decision diagram, the artefacts table.

### Limitations

- **The layer list and the method are the book's proposal** (`ch30-lim3`), not documented by Pi.
- **The method is unvalidated against a set of real failures** — a heuristic, not a procedure.
- **The `onPayload` snippet is `documented-not-run`** — a real `pi-ai` API, not exercised here.
- **No real model is called** — no diagnosis of any particular failure is made.

### Optional work that was NOT run

- The `onPayload` payload-logging snippet (documented, not run).
- `/debug`, `/bug`, `/session`, `/hotkeys`, `pi --export` (terminal artefacts, not run here).
- The bare-`Agent` replacement step (chapter 37 has the bare agent; this notebook does not re-run it).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
